# MFCC et FILTRAGE OUTLIERS

In [ ]:
import os
import pandas as pd
import numpy as np
import subprocess
from tqdm import tqdm
import warnings
from scipy.spatial.distance import cdist
from pathlib import Path

warnings.filterwarnings('ignore')

# DEBUT CHARGEMENT
print("Chargement des données...")
df_en = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/english/all_aligned_clean_english.csv", sep=r"\s+")
df_fr = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/french/all_aligned_clean_french.csv", sep=r"\s+")
df_en['language'] = 'english'
df_fr['language'] = 'french'
df_items = pd.concat([df_en, df_fr])

df_tests = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/human_and_models.csv", low_memory=False)

# Construction de item_triplet 
df_tests['item_triplet'] = df_tests['language'].astype(str) + "_" + \
                            df_tests['TGT_item'].astype(str) + "_" + \
                            df_tests['OTH_item'].astype(str) + "_" + \
                            df_tests['X_item'].astype(str)

# Outliers connus, liste hardcodée
outliers_total = {34, 35, 72, 76, 77, 79, 82, 86, 92, 123, 140, 141, 159, 161, 179, 180}

# df_tests 
df_tests = df_tests[~df_tests['individual'].isin(outliers_total)].copy()

# triplets valides 
triplets_valides = set(df_tests['item_triplet'])

# df_triplets = liste des triplets uniques à calculer
df_triplets = df_tests.drop_duplicates(subset=['filename', 'language']).copy()

print(f"Outliers supprimés ({len(outliers_total)}) : {sorted(outliers_total)}")
print(f"Triplets à calculer : {len(df_triplets)}")
print("Données chargées.")


def get_file_info(item_index, lang):
    row = df_items[(df_items['index'] == item_index) & (df_items['language'] == lang)].iloc[0]
    return f"../../data-wav/{lang}/1s/{row['#file']}.wav", row['onset'], row['offset']

# DOSSIERS DE SAUVEGARDE 
SAVE_DIR = Path().resolve() / "matrices_mfcc_sauvegardees"

stats_matrices = {'disque': 0, 'kaldi': 0}

def load_matrice(filepath, lang, item_id, onset, offset):
    """
    Vérifie si la matrice découpée existe déjà sur le disque.
    Si OUI, la charge instantanément.
    """
    nom_fichier = f"{item_id}.npy"
    chemin_sauvegarde = SAVE_DIR / lang / nom_fichier
    stats_matrices['disque'] += 1
    return np.load(chemin_sauvegarde)

Chargement des données...
Outliers supprimés (16) : [34, 35, 72, 76, 77, 79, 82, 86, 92, 123, 140, 141, 159, 161, 179, 180]
Triplets à calculer : 5202
Données chargées.


# DTW

In [ ]:
# 3. LE DTW 


def accelerated_dtw_exp(x, y, dist='cosine'):

    r, c = len(x), len(y)
    D0 = np.zeros((r + 1, c + 1))
    D0[0, 1:] = np.inf
    D0[1:, 0] = np.inf
    D1 = D0[1:, 1:]
    A = cdist(x, y, dist)
    D0[1:, 1:] = A.copy()
    C = D1.copy()
    for i in range(r):
        for j in range(c):
            min_list = [D0[i, j]]
            for k in range(1, 2):  # warp=1
                min_list += [D0[min(i+k, r-1), j], D0[i, min(j+k, c-1)]]
            D1[i, j] += min(min_list)
    return D1[-1, -1] / float(max(x.shape[0], y.shape[0])) 

def calculer_delta_machine(mfcc_X, mfcc_TGT, mfcc_OTH):
    if mfcc_X.shape[0] < 2 or mfcc_TGT.shape[0] < 2 or mfcc_OTH.shape[0] < 2:
        return np.nan
    try:
        norm_TGT = accelerated_dtw_exp(mfcc_TGT, mfcc_X)
        norm_OTH = accelerated_dtw_exp(mfcc_OTH, mfcc_X)
        return (norm_OTH - norm_TGT)
    except:
        return np.nan

# POW

In [ ]:
import sys

sys.path.append('../../Partial-Ordered-Wasserstein-Distance/src')
from pow.pow import partial_order_wasserstein

def calculer_delta_pow(mfcc_X, mfcc_TGT, mfcc_OTH, order_reg, omega=0.001):
    """
    Calcule la différence de distance POW brute avec masse partagée m*.
    """
    # 1. Construction des matrices 
    
    D_TGT = cdist(mfcc_TGT, mfcc_X, 'cosine')
    D_OTH = cdist(mfcc_OTH, mfcc_X, 'cosine')
    m_optimal = 0.95

    # 2. Calcul 
    norm_TGT = partial_order_wasserstein(M=D_TGT, order_reg=order_reg, m=m_optimal, 
                                         return_dist=True, ot_algo="sinkhorn")
    norm_OTH = partial_order_wasserstein(M=D_OTH, order_reg=order_reg, m=m_optimal, 
                                         return_dist=True, ot_algo="sinkhorn")
    
    delta = norm_OTH - norm_TGT
    return delta, m_optimal


# ANALYSE

In [ ]:
# 1. INITIALISATION DU LAMBDA GLOBAL
lambda_dataset = 5

# 2. LANCEMENT DU CALCUL
deltas_calcules = []
print("Lancement du calcul complet...")

for index, row in tqdm(df_triplets.iterrows(), total=len(df_triplets), desc="Progression"):
    lang = row['language']
    path_TGT, on_TGT, off_TGT = get_file_info(row['TGT_item'], lang)
    path_OTH, on_OTH, off_OTH = get_file_info(row['OTH_item'], lang)
    path_X, on_X, off_X = get_file_info(row['X_item'], lang)
    
    mfcc_TGT = load_matrice(path_TGT, lang, row['TGT_item'], on_TGT, off_TGT)
    mfcc_OTH = load_matrice(path_OTH, lang, row['OTH_item'], on_OTH, off_OTH)
    mfcc_X   = load_matrice(path_X, lang, row['X_item'], on_X, off_X)
    
    delta_dtw = calculer_delta_machine(mfcc_X, mfcc_TGT, mfcc_OTH)
    delta_pow, m_optimal = calculer_delta_pow(mfcc_X, mfcc_TGT, mfcc_OTH, order_reg=lambda_dataset) 
    
    deltas_calcules.append({
        'filename': row['filename'],
        'language': lang,
        'Delta_Machine_DTW_PERSO': delta_dtw,
        'Delta_Machine_POW': delta_pow,
        'Optimal_m': m_optimal
    })



Lancement du calcul complet...


Progression: 100%|██████████| 5202/5202 [00:33<00:00, 156.44it/s]


In [ ]:
df_resultats_machine = pd.DataFrame(deltas_calcules)
df_final = pd.merge(df_tests, df_resultats_machine, on=['filename', 'language'], how='left')

df_final['Prediction_MFCC'] = (df_final['MFCC'] > 0).astype(int)
df_final['Prediction_POW'] = (df_final['Delta_Machine_POW'] > 0).astype(int)

chemin_csv = SAVE_DIR / "resultats_pow_WO_OT.csv"
df_final.to_csv(chemin_csv, index=False)